In [1]:
import pandas as pd
from datetime import datetime, timedelta
from math import floor

In [ ]:
DIRECTORY = "some_path"
FILE_NAME = "MasterSheet.xlsx"
SHEET_NAME = "StudentsLog"

PITCH_LENGTH = 6.0      # minutes
QA_LENGTH = 1.5         # minutes
SESSION_START = datetime(2026, 5, 29, 13, 20)
TALK_START, TALK_END = datetime(2026, 5, 29, 14, 00), datetime(2026, 5, 29, 15, 00)
BREAK_START, BREAK_END = datetime(2026, 5, 29, 15, 00), datetime(2026, 5, 29, 15, 15)

In [10]:
df = pd.read_excel(f"{DIRECTORY}/{FILE_NAME}", sheet_name=SHEET_NAME)
df["-> SG"] = df["-> SG"].astype(str)
df = df.loc[(~ df["-> SG"].str.contains("withdrew")) & (~ df["-> SG"].str.contains("audit"))]
df = df.rename(mapper=lambda s: s.strip().lower(), axis=1)
df = df[["name",
         "id",
         "school",
         "dept",
         "course",
         "discord",
         "github",
         "team mates",
         "constraint"]]
df["name"] = df["name"].map(lambda s: s.strip())
df = df.loc[(~df["school"].str.contains("NCU"))
            & (~df["team mates"].str.contains("withdrew"))
            & (~df["team mates"].str.contains("auditor"))]
df["team mates"] = df["team mates"].map(lambda s: "" if isinstance(s, float) else s)
df["team mates"] = df["team mates"].map(lambda s: s.split(", "))
df = df.reset_index(drop=True)

In [11]:
max_before_talk = floor((TALK_START - SESSION_START) / timedelta(minutes=(PITCH_LENGTH + QA_LENGTH)))
max_after_talk = df.shape[0] - max_before_talk 

slots = [SESSION_START + timedelta(minutes=(PITCH_LENGTH + QA_LENGTH) * i) for i in range(max_before_talk)]
slots.extend(BREAK_END + timedelta(minutes=(PITCH_LENGTH + QA_LENGTH) * i) for i in range(max_after_talk))
assert len(slots) == df.shape[0]

schedule = pd.DataFrame({"TIME": slots})
schedule = schedule.map(lambda t: t.strftime("%H:%M:%S"))

In [14]:
def get_groups(df):
    groups = [tuple(sorted([df.iloc[i]["name"]] + df.iloc[i]["team mates"]))
                                                for i in range(df.shape[0])]
    groups = set(groups)
    groups = [tuple([s for s in grp if s!= ""]) for grp in groups]
    return groups

df_first = df.loc[df["constraint"]==1].copy()
df_others = df.loc[df["constraint"]!=1].copy()
groups_first = get_groups(df_first)
groups_others = get_groups(df_others)

table_first = pd.DataFrame({"GROUP": groups_first})
table_others = pd.DataFrame({"GROUP": groups_others}).sample(frac=1, random_state=42)

table_first = pd.concat([table_first.loc[table_first["GROUP"].copy().astype(str).str.contains("Nandini")],
                         table_first.loc[~table_first["GROUP"].copy().astype(str).str.contains("Nandini")].sample(frac=1,
                                                                                                                  random_state=42)])
table = pd.concat([table_first, table_others]).reset_index(drop=True)
table = pd.concat([schedule.iloc[:len(table)], table], axis=1)
table = table.T
table.insert(max_before_talk,
             max_before_talk,
             [TALK_START.strftime("%H:%M:%S"), "DR. JEREMY'S TALK"],
             allow_duplicates=True)
table.insert(max_before_talk + 1,
             max_before_talk + 1,
             [BREAK_START.strftime("%H:%M:%S"), "SHORT BREAK"],
             allow_duplicates=True)
table = table.T
table = table.reset_index(drop=True)
              

In [15]:
def format(cell):
    if isinstance(cell, tuple):
        if len(cell) == 1:
            return cell[0]
        else:
            return " + ".join(cell)
    elif isinstance(cell, str):
        return cell

table["GROUP"] = table["GROUP"].map(format)
# display(table)

In [16]:
table.to_excel(f"{DIRECTORY}/pitch_schedule.xlsx", index=False)